# Taxonomy Classification Batch Test

이 노트북은 `run_taxonomy_classification_batch.py`를 Databricks에서 검증하기 위한 테스트용입니다.

실행 순서:
1. repo 최신 pull
2. runner import / reload
3. 선택 그룹 batch 실행
4. classification detail 결과 및 progress checkpoint 확인


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)


In [ ]:
%sh
cd /Workspace/Users/jungryo.lee@lge.com/prj_TV_voc && git pull

In [ ]:
PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import pipeline.run_taxonomy_classification_batch as batch_runner

importlib.reload(config_loader)
importlib.reload(batch_runner)

from common.config_loader import get_log_table, get_output_table, load_config
from pipeline.run_taxonomy_classification_batch import run_taxonomy_classification_batch

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")

CLASSIFICATION_DETAIL_TABLE = get_output_table(config, "classification_detail")
PIPELINE_PROGRESS_TABLE = get_log_table(config, "pipeline_progress")


In [ ]:
MODEL_KEY = "gpt_55"
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = None
TARGET_SC_MEASUREMENT = 1
LIMIT_GROUP_COUNT = 3
MAX_ROWS_PER_GROUP = 100

batch_result = run_taxonomy_classification_batch(
    spark,
    config=config,
    model_key=MODEL_KEY,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    limit_group_count=LIMIT_GROUP_COUNT,
    max_rows_per_group=MAX_ROWS_PER_GROUP,
    use_llm_fallback=True,
    resume_from_checkpoint=True,
    cleanup_checkpoint_on_success=True,
    print_progress=True,
)

print({
    "group_count": batch_result["group_count"],
    "processed_group_count": batch_result["processed_group_count"],
    "skipped_group_count": batch_result["skipped_group_count"],
    "classification_count": batch_result["classification_count"],
    "total_row_count": batch_result["total_row_count"],
    "total_overall_count": batch_result["total_overall_count"],
    "total_topic_count": batch_result["total_topic_count"],
    "total_others_count": batch_result["total_others_count"],
    "total_llm_used_count": batch_result["total_llm_used_count"],
    "checkpoint_key": batch_result["checkpoint_key"],
})

In [ ]:
display(
    spark.createDataFrame(batch_result["classification_summaries"])
    .orderBy(F.col("others_ratio").desc(), F.col("cate_2_depth").asc())
)

In [ ]:
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]

detail_df = (
    spark.table(CLASSIFICATION_DETAIL_TABLE)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
)

display(
    detail_df.groupBy("cate_2_depth", "pred_topic")
    .agg(F.count("*").alias("memo_cnt"))
    .orderBy(F.col("cate_2_depth").asc(), F.col("memo_cnt").desc())
)

In [ ]:
display(
    detail_df.select(
        "cate_2_depth",
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(30)
)

In [ ]:
display(
    spark.table(PIPELINE_PROGRESS_TABLE)
    .where(F.col("checkpoint_key") == batch_result["checkpoint_key"])
    .orderBy(F.col("created_at").desc())
)